# ESZA019 - Visão Computacional
## Laboratório 7 - Introdução às redes CNN
*Convolutional Neural Networks* para Reconhecimento em Imagens

Neste roteiro, para fins didáticos, utilizaremos o **Google Colab Interativo** o framework  **TensorFlow / Keras**, projetado para alunos de graduação em Engenharia e Computação.

Fundamentos: **Szeliski, R. *Computer Vision: Algorithms and Applications (2nd Ed.)*, Capítulo 5.4.**

Aplicação prática em robótica móvel/manipulação: integração de visão em tempo real usando a webcam.

O relatório da equipe deverá ser feito no **Jupyter notebook**.

---

## 🎯 Objetivos de Aprendizagem
1. Compreender a transição dos **Filtros Espaciais Clássicos** (Sobel, Gaussiano) para os **Filtros Aprendidos (CNN)**.
2. Construir e treinar uma CNN compacta em **TensorFlow/Keras** para reconhecimento de objetos.
3. Inspecionar os *Feature Maps* das camadas convolucionais para "abrir a caixa-preta" da rede.
4. Testar a rede em tempo real conectando um sensor de visão (Webcam) diretamente no Colab.

---

## 📚 Conexão Teórica: Szeliski Cap. 5.4
No processamento de imagem tradicional, definimos manualmente os pesos do filtro. Por exemplo, para detectar bordas verticais para navegação de um robô, usamos um filtro Sobel $3 \times 3$:

$$H = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$$

Como visto em **Szeliski 5.4**, uma **Camada Convolucional** aplica múltiplos kernels $K \times K \times C$ em paralelo, mas com uma diferença fundamental: **os valores do kernel são parâmetros ajustáveis $(\theta)$ aprendidos via Gradiente Descendente e Backpropagation**.


In [ ]:
## IMPORTANTE: Antes de iniciar, configure no menu do Colab:
# [Edit] -> [Notebook settings] -> ()T4 GPU

In [ ]:
## 1. Importação de Bibliotecas
# Importação das bibliotecas essenciais
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print(f"Versão do TensorFlow: {tf.__version__}")
# Verificar se a GPU está ativa (fundamental para aceleração em robótica)
gpu_available = len(tf.config.list_physical_devices('GPU')) > 0
print(f"Aceleração por GPU Ativa: {gpu_available}")

## Parte 1. Filtro Manual (Sobel) vs. Conceito de Kernel Aprendido
Antes de construir a rede **CNN**, aplicar um **filtro de convolução** manual em uma imagem de teste para visualizar a **extração de características**.


In [ ]:
## 2. Convolução Clássica com OpenCV/NumPy
# Carregando uma imagem de exemplo do dataset CIFAR-10
(x_train_raw, y_train_raw), _ = tf.keras.datasets.cifar10.load_data()
sample_img = x_train_raw[0] # Imagem 32x32 RGB

# Converter para escala de cinza
gray_img = cv2.cvtColor(sample_img, cv2.COLOR_RGB2GRAY)

# Definindo o Filtro Sobel X (Bordas Verticais - Szeliski 5.4.1)
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)

# Aplicando a Convolução
edges_x = cv2.filter2D(gray_img, -1, sobel_x)

# Visualização
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(sample_img)
plt.title("Imagem Original (32x32 RGB)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(edges_x, cmap='gray')
plt.title("Convolução Manual: Filtro Sobel X")
plt.axis('off')
plt.show()

**Elabore:** Numa nova célula abaixo, código para gerar mais exemplos desta filtragem, sendo pelo menos três imagens diferentes do dataset CIFAR-10, por aluno da equipe. Dica: modifique o índice no comando: sample_img = x_train_raw[0]

**Analise os resultados e responda com suas próprias palavras:** sobre a diferença entre processamento de imagem clássico e aprendizado profundo:

* **1:** Foi aplicado o filtro clássico de Sobel para detectar bordas verticais. Qual é a principal limitação matemática e prática de utilizar apenas filtros codificados manualmente (como Sobel ou Canny) no sistema de visão de um robô que navega em ambientes externos?

* **2:** Com base no capítulo 5.4 de Szeliski, explique com suas palavras: o que exatamente a rede neural está "aprendendo" durante a fase de treinamento em uma camada convolucional que substitui a necessidade de criarmos o filtro manualmente?


## Parte 2: Pipeline de Dados de Percepção

### . Preparação do Dataset (CIFAR-10)
Utilizaremos o dataset CIFAR-10 contendo 10 classes (incluindo veículos como *carro*, *caminhão*, *avião* e *navio*, relevantes para robótica móvel).

In [ ]:
## 3. Download e Pré-processamento
# Carregar CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalização dos píxeis para o intervalo [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

class_names = ['Aviao', 'Automovel', 'Passaro', 'Gato', 'Cervo',
               'Cachorro', 'Sapo', 'Cavalo', 'Navio', 'Caminhao']

# Visualizar amostras
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.tight_layout()
plt.show()

**Analise os resultados e responda com suas próprias palavras:** sobre  a preparação dos dados sensoriais e representação matricial:

* **3:** Na preparação do dataset, os pixels foram divididos por $255.0$. Quanto à otimização matemática (cálculo de gradientes), por que essa normalização é necessária para o treinamento da rede neural do robô?

* **4:** O dataset CIFAR-10 possui apenas imagens de ($32 \times 32$ pixels). Se o robô utilizar câmera 4K ($3840 \times 2160$ pixels), o que aconteceria com a quantidade de parâmetros da primeira camada densa (*Flatten* seguida de *Dense*) e como isso impactaria o hardware embarcado? (Apresente exemplos)

## Parte 3: Construção da CNN em Keras

### . Arquitetura da Rede Convolucional (RobotVisionNet)
A arquitetura segue o padrão clássico descrito por Szeliski:
1. **Camadas Convolucionais (`Conv2D`):** Extração de mapas de características.
2. **Camadas de Max Pooling (`MaxPooling2D`):** Redução dimensional e garantia de **invariância a pequenas translações** do objeto no campo de visão da câmera.
3. **Camadas Densas (`Dense`):** Tomada de decisão/classificação final.


In [ ]:
## 4. Definição do Modelo Keras

def build_robot_cnn(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        # Bloco Convolucional 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape, name='conv_1'),
        layers.MaxPooling2D((2, 2), name='pool_1'),

        # Bloco Convolucional 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv_2'),
        layers.MaxPooling2D((2, 2), name='pool_2'),

        # Classificador (Fully Connected)
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='fc_1'),
        layers.Dropout(0.3, name='dropout'),
        layers.Dense(num_classes, activation='softmax', name='output')
    ])
    return model

model = build_robot_cnn()
model.summary()

**Analise os resultados e responda com suas próprias palavras:** sobre a arquitetura da rede e suas propriedades espaciais.

* **5:** Na arquitetura construída, por que as camadas convolucionais (Conv2D) são posicionadas no início do modelo e as camadas densas (Dense) apenas no final, e não o contrário? Qual o papel funcional de cada um desses blocos metodológicos?

## Parte 4: Compilação e Treinamento

### . Treinamento da Rede
Definimos a função de perda (`sparse_categorical_crossentropy`) e o otimizador (`Adam`).


In [ ]:
## 5. Execução do Treinamento

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Treinamento por 10 épocas
history = model.fit(x_train, y_train,
                    epochs=10,
                    validation_data=(x_test, y_test),
                    batch_size=64)

**Analise os resultados e responda com suas próprias palavras:** sobre a dinâmica do aprendizado, as métricas e a função de custo.

* **6:** Durante as 10 épocas de treinamento, o gráfico gerado mostra os valores da *Loss* (Perda). Defina o que a função de perda mede. Se o gráfico demonstrar que a perda de treinamento continua caindo rumo a zero, mas a perda de validação estaciona ou começa a subir, qual fenômeno está ocorrendo e como ele afetaria o robô na prática?


## Parte 5: Diagnóstico e Matriz de Confusão

### . Avaliação do Desempenho
Em Engenharia Robótica, é necessário identificar falhas específicas: se o robô confundir um *automóvel* com um *caminhão*, o impacto na navegação pode ser tolerável; se confundir um *pedestre/animal* com um *objeto inanimado*, o erro é crítico.

In [ ]:
## 6. Gráficos e Matriz de Confusão

# Curvas de Aprendizado
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Treino')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.title('Acurácia da Visão do Robô')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.xlabel('Época')
plt.ylabel('Perda (Loss)')
plt.title('Perda durante o Treinamento')
plt.legend()
plt.show()

# Matriz de Confusão
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predito pelo Robô')
plt.ylabel('Classe Real')
plt.title('Matriz de Confusão - Erros e Acertos da Visão')
plt.show()

**Analise os resultados e responda com suas próprias palavras:** sobre as métricas de forma crítica com foco na tomada de decisão do robô.

* **7:** Observe a Matriz de Confusão gerada pelo seu modelo. Quais foram as duas classes com o maior índice de falsos positivos. Do ponto de vista da semântica das imagens, por que a rede cometeu esse erro específico?

* **8:** Se hipoteticamente este modelo é implantado no sistema de frenagem de emergência de um robô de patrulha. Errar a classe "Automóvel" classificando-a como "Caminhão" tem o mesmo peso de gravidade que classificar um "Cachorro" como um "Caminhão"? Como a matriz de confusão nos ajuda a auditar a segurança do robô?

## Parte 6: Visualizando as Características Aprendidas (Szeliski 5.4.6)

### . Visualização dos *Feature Maps*
**Objetivo:** Extrair as saídas da primeira camada convolucional (`conv_1`) após o treinamento para comprovar que a rede aprendeu automaticamente detectores de bordas, orientação e contraste!

Atenção: os mapas de características (*feature_maps*) representam as ativações dos diferentes filtros (ou kernels) aprendidos pela primeira camada convolucional (conv_1) da rede. Estes elementos serão progressivamente combinados e refinados nas camadas mais profundas para formar representações complexas que permitirão à rede distinguir entre as dez classes diferentes do dataset CIFAR-10.

In [ ]:
## 7. Inspeção de Ativações da Camada Convolucional

model.build(input_shape=(None, 32, 32, 3))
# Forçar o modelo a ser 'chamado' para garantir que seu grafo esteja construído
dummy_input = tf.zeros((1, 32, 32, 3))
_ = model(dummy_input)

# Criar modelo intermediário para extrair a saída da camada 'conv_1'
activation_model = tf.keras.Model(inputs=model.layers[0].input, outputs=model.get_layer('conv_1').output)

# Obter ativações para uma imagem de teste
sample_input = np.expand_dims(x_test[0], axis=0);
feature_maps = activation_model.predict(sample_input) # Formato: (1, 32, 32, 32)

plt.figure(figsize=(12, 6))
for i in range(16): # Exibir os primeiros 16 canais aprendidos
    plt.subplot(4, 4, i+1)
    plt.imshow(feature_maps[0, :, :, i], cmap='viridis')
    plt.axis('off')
    plt.title(f'Canal {i+1}')
plt.suptitle('Mapas de Características Aprendidos pela Camada conv_1', fontsize=14)
plt.show()

**Analise os resultados e responda com suas próprias palavras:** sobre a interpretabilidade de redes neurais.

* **9:** Nas saídas (ativações) da camada conv_1, alguns canais destacam bordas enquanto outros destacam texturas. Caso visualizasse os *Feature Maps* da última camada convolucional antes do classificador, seria possível encontrar características mais simples (como linhas e pontos) ou mais complexas (como formatos abstratos do objeto inteiro)? Explique o motivo anatômico da CNN.

## Parte 7: Aplicação Prática com Webcam ao Vivo

### . Teste de Percepção em Tempo Real via Webcam
Agora conectar a câmera do seu computador! Mostrar objetos reais (como brinquedos de carros, miniaturas ou imagens no celular) para testar a predição da sua rede neural em tempo real.

In [ ]:
## 8. Bridge JavaScript/Python para Captura de Webcam no Colab

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Função JavaScript para acessar a webcam e capturar um frame
def capture_webcam_frame():
    js = Javascript('''
    async function takePhoto() {
      const div = document.createElement('div');
      const video = document.createElement('video');
      const capture = document.createElement('button');

      capture.textContent = '📸 Capturar Frame e Classificar';
      capture.style.padding = '10px 20px';
      capture.style.fontSize = '16px';
      capture.style.backgroundColor = '#008CBA';
      capture.style.color = 'white';
      capture.style.border = 'none';
      capture.style.borderRadius = '5px';
      capture.style.cursor = 'pointer';
      capture.style.marginTop = '10px';

      video.style.display = 'block';
      video.style.borderRadius = '8px';

      let stream;
      try {
        stream = await navigator.mediaDevices.getUserMedia({video: true});
      } catch (err) {
        throw new Error("Permissão da câmera negada. Por favor, permita o acesso na barra do navegador.");
      }

      document.body.appendChild(div);
      div.appendChild(video);
      div.appendChild(capture);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getTracks().forEach(track => track.stop());
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(js)
    data = eval_js('takePhoto()')

    binary = b64decode(data.split(',')[1])
    img_pil = Image.open(io.BytesIO(binary))
    img_np = np.array(img_pil)
    return img_np

try:
    print("Aguardando interação com a câmera...")
    raw_frame = capture_webcam_frame()

    # Pré-processamento
    img_resized = cv2.resize(raw_frame, (32, 32))
    img_normalized = img_resized.astype('float32') / 255.0
    img_tensor = np.expand_dims(img_normalized, axis=0)

    # Inferência
    prediction = model.predict(img_tensor)
    class_idx = np.argmax(prediction[0])
    confidence = prediction[0][class_idx] * 100

    # Exibição
    plt.figure(figsize=(6, 6))
    plt.imshow(raw_frame)
    plt.title(f"Predição do Robô: {class_names[class_idx]} ({confidence:.1f}%)",
              fontsize=14, fontweight='bold', color='darkblue')
    plt.axis('off')
    plt.show()
except Exception as e:
    print(f"Erro ao acessar a câmera: {e}")

**Analise os resultados e responda com suas próprias palavras:**   robustez, latência e transferência do modelo para o mundo físico.

* **10:** O fenômeno de *Domain Shift* (mudança de domínio) ocorre quando os dados em que o modelo foi treinado diferem substancialmente dos dados em que ele é testado. Cite pelo menos três fatores físicos do ambiente da sua webcam que diferem das imagens originais do CIFAR-10 e que fizeram a acurácia do seu modelo cair ao vivo.

* **11:** Quando o código captura um frame da sua webcam e realiza a inferência (model.predict), isso consome um certo tempo em milissegundos. Se um robô móvel avança a $2 \text{ m/s}$ e o seu pipeline de visão da webcam demora $500 \text{ ms}$ para processar a imagem e classificar um obstáculo, quantos metros o robô terá percorrido "às cegas" entre o momento da captura e a decisão final? Comente a relação entre complexidade da CNN e segurança mecânica.

---

## 💡 Parte 8: QUESTÕES:

1. ** *Trade-off* (compromisso) entre Latência vs. Acurácia:**

Questão 1:
* *"Se o robô operar a 30 FPS, ele dispõe de apenas 33 ms por ciclo para capturar a imagem, processar a CNN e enviar o comando aos motores. Qual é o tempo de inferência da rede que acabamos de treinar?"*

Dica: (`time.time()` antes e depois do `model.predict`).


2. **A "Ilusão" da Acurácia em Ambiente Controlado:**

Questão 2:
*  Com a webcam, a acurácia poderá ser bem menor em relação ao conjunto de teste do CIFAR-10.
* Discuta os fatores: **iluminação ambiente**, **fundo complexo**, **mudança de escala**. Apresente exemplos com imagens de cada um destes fatores.


## Parte 9:  🚀 Desafio Prático

Adicionar uma célula ao final do notebook e implementarem as seguintes melhorias:

1. **Aumento de Dados (*Data Augmentation*):**
Adicionar camadas de transformação aleatória na entrada do modelo Keras para tornar o robô mais robusto a variações de orientação e iluminação na câmera:

In [ ]:
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

2. **Filtro de Rejeição por Confiança (Safety Threshold):**
Modificar o código da webcam para que o robô emita um aviso de *"Objeto Não Identificado / Incerto"* caso a confiança da maior probabilidade seja inferior a **60%** ($\text{Confiança} < 0.60$), prevenindo ações desastrosas do manipulador mecânico.

- - -

### 10) Relatório: Elaborar o relatório em formato **jupyter** e hospedar no github, conforme instruções em aulas anteriores.

Além de texto e códigos, o relatório deve conter as imagens e os videos.

O relatório deverá conter pelo menos os seguintes tipos de Seções:
- Título do relatório
- Nome completo dos autores do relatório
- Data de realização dos experimentos
- Data de publicação do relatório
- Introdução – apresentando o que será descrito e relatado
- Fundamentação Teórica - introdução ao assunto a ser estudado nos experimentos
- Procedimentos experimentais – explicando como realizar e executar as atividades
- Análise e discussão dos estudos realizados
- Conclusões
- Referências consultadas e indicadas.

Cada relatório deverá ser colocado numa pasta separada, junto com os arquivos pertinentes.
A página HTML da equipe deverá conter um índice das aulas de laboratório, com um link para cada relatório.





### ATENÇÃO ESPECIAL:

1- O Cnpq lançou uma portaria 2664/2026, que discute de forma mais ampla a integridade da pesquisa científica no CNPq, e possui algumas diretrizes (replicada abaixo) para o uso da IA.

Link: http://memoria2.cnpq.br/web/guest/view/-/journal_content/56_INSTANCE_0oED/10157/23142775
"
c) declarar o uso de ferramentas de Inteligência Artificial Generativa - IAG, de qualquer espécie e em qualquer fase do desenvolvimento da pesquisa (concepção, redação, análise de dados, submissão) especificando nos respectivos textos e exposições eletrônicas, a ferramenta utilizada e a finalidade;

d) é vedada a submissão de conteúdo gerado por IAG como se fosse de autoria humana, sendo os autores integralmente responsáveis pelo conteúdo final, inclusive por eventuais plágios ou imprecisões geradas pela IAG;

e) é vedada a inserção de projetos de pesquisa de terceiros em ferramentas de IAG para elaboração de pareceres científicos;

f) responsabilizar-se integralmente pelo conteúdo final da pesquisa, inclusive por eventuais plágios ou imprecisões geradas pela IAG;
"

Portanto, se for seu caso, faça tais menções de IAG no seu relatório de forma rigorosa.

- - - -